# 5. Modele typu Decoder

Modele typu **Decoder** w przeciwieństwie do modeli encoderowych nie koncentrują się przede wszystkim na pełnym zrozumieniu całego wejścia naraz, ale na **przewidywaniu kolejnych tokenów**. Ich głównym celem jest generowanie tekstu krok po kroku: od lewej do prawej.

Najbardziej klasyczne przykłady to rodziny modeli:
- GPT / GPT-2,
- Llama i jej warianty,
- Mistral,
- Pythia,
- TinyLlama,
- SmolLM.

Modele decoderowe są szczególnie skuteczne w zadaniach:
- generowania tekstu,
- uzupełniania promptu,
- odpowiadania na pytania,
- streszczania i parafrazowania,
- pisania kodu,
- prowadzenia dialogu,
- prostych zadań typu instruction following.

W praktyce pomagają one odpowiedzieć na pytanie:

**„Jaki token powinien pojawić się jako następny?”**

## Wymagania

Konto (w razie napotkania ograniczeń): [huggingface.co](https://huggingface.co/)

- `transformers` - modele, tokenizery i pipeline’y z Hugging Face
- `datasets` - zbiory danych i wygodne przetwarzanie danych
- `evaluate` - liczenie metryk
- `accelerate` - prostsze uruchamianie modeli na CPU/GPU
- `sentencepiece` - wymagane przez część tokenizerów
- `scikit-learn` - narzędzia pomocnicze

In [ ]:
# Instalacja pakietów w Colabie
!pip -q install transformers datasets evaluate accelerate sentencepiece scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.2 MB/s eta 0:00:00


## 1. Ekosystem Hugging Face

W ekosystemie Hugging Face najczęściej pracujemy z trzema elementami:

- `tokenizer` - zamienia tekst na tokeny i identyfikatory liczbowe,
- `model` - wykonuje obliczenia na tych danych,
- `pipeline` - wygodna nakładka upraszczająca inferencję.

Dla modeli decoderowych najczęściej spotkasz:
- `AutoTokenizer`
- `AutoModelForCausalLM`
- `pipeline("text-generation")`

Słowo **causal** oznacza tutaj, że model w trakcie przewidywania patrzy tylko na to, co było wcześniej.

### Ręczne wczytywanie modelu

- `AutoTokenizer` wczytuje tokenizer odpowiedni dla wybranego modelu,
- `AutoModelForCausalLM` wczytuje model przygotowany do zadania przewidywania kolejnych tokenów,
- `from_pretrained(...)` pobiera gotowy checkpoint.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_id, token="")
model = AutoModelForCausalLM.from_pretrained(model_id)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
text = "Transformers are very useful for"
inputs = tokenizer(text, return_tensors="pt")
print(inputs)

{'input_ids': tensor([[41762,   364,   389,   845,  4465,   329]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])}


To zwraca słownik z danymi wejściowymi, np.:

- `input_ids` - numery tokenów,
- `attention_mask` - informacja, które pozycje należą do prawdziwego tekstu.

Możemy też podejrzeć tokeny:

In [ ]:
tokens = tokenizer.tokenize(text)
print(tokens)

['Transform', 'ers', 'Ġare', 'Ġvery', 'Ġuseful', 'Ġfor']


### Pipeline

`pipeline("text-generation")` to najprostszy sposób pracy z modelem decoderowym, bo:

- sam pobiera model i tokenizer,
- sam przygotowuje dane wejściowe,
- sam wykonuje generację,
- zwraca gotowy tekst.

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model="distilgpt2")
result = generator("Ala ma kota, a jutro", max_new_tokens=20, do_sample=True)
print(result[0]["generated_text"])

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=20) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Ala ma kota, a jutro, a jutro."



You can also follow him on Twitter and on Facebook


## 2. Import wymaganych bibliotek

In [ ]:
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)

## 3. Określenie urządzenia

In [ ]:
device = 0 if torch.cuda.is_available() else -1
print("CUDA available:", torch.cuda.is_available())
print("Device for pipeline:", device)

CUDA available: True
Device for pipeline: 0


## 4. Generowanie tekstu - podstawowy przykład

Modele decoderowe są trenowane przez **Causal Language Modeling**.

W uproszczeniu wygląda to tak:

jeżeli model dostaje fragment:

> `Alicja przecięła balon i`

to uczy się przewidywać, jaki token pojawi się potem, np.:
- `uciekło`
- `powietrze`
- `balon`
- `pękł`

Model nie widzi „przyszłości”, tylko wszystko, co było wcześniej. To kluczowa różnica względem klasycznych encoderów.

### Przykład i zadanie

Uruchom poniższy kod i porównaj wyniki dla modeli:
- `distilgpt2`
- `gpt2`
- `EleutherAI/pythia-160m`

Następnie dopisz własne 3 prompty:
- jeden po angielsku,
- jeden po polsku,
- jeden domenowy (np. informatyka, edukacja, medycyna).

Sprawdź, czy model:
- trzyma temat,
- kończy logicznie zdanie,
- przechodzi na inny język,
- zaczyna powtarzać tokeny.

In [ ]:
def run_generation(model_id, prompt, max_new_tokens=40, temperature=0.8, top_p=0.95):
    generator = pipeline(
        "text-generation",
        model=model_id,
        tokenizer=model_id,
        device=device
    )

    outputs = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        return_full_text=True
    )

    print(f"\nMODEL: {model_id}")
    print("PROMPT:", prompt)
    print("WYNIK:")
    print(outputs[0]["generated_text"])

models = [
    "distilgpt2",
    "gpt2",
    "EleutherAI/pythia-160m",
]

prompt_en = "Transformers are useful because"

for model_id in models:
    run_generation(model_id, prompt_en)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



MODEL: distilgpt2
PROMPT: Transformers are useful because
WYNIK:
Transformers are useful because they are easy to use.




































Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



MODEL: gpt2
PROMPT: Transformers are useful because
WYNIK:
Transformers are useful because they make our world look much cleaner. They're easy to work with, have many features that make it easy to develop code and test it, and are the most powerful tools for building great apps.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



MODEL: EleutherAI/pythia-160m
PROMPT: Transformers are useful because
WYNIK:
Transformers are useful because of how they are built.

These examples are from the documentation, which is available online at
<http://www.w3.org/TR/html401-functions/#func-


## 5. Następny token - co model przewiduje naprawdę?

Wygenerowany tekst nie bierze się „znikąd”. W każdej chwili model wylicza rozkład prawdopodobieństwa nad słownikiem i ocenia, który token powinien być następny.

To oznacza, że dla promptu:

> `Deep learning is`

model nie „zna całego zdania”, tylko ocenia:
- jaki token ma największe prawdopodobieństwo teraz,
- potem dokleja go do wejścia,
- ponownie liczy rozkład dla kolejnego kroku,
- i tak aż do końca generacji.

In [ ]:
import torch.nn.functional as F

model_id = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

prompt = "Deep learning is"
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits[:, -1, :]
    probs = F.softmax(logits, dim=-1)

top_k = 10
top_probs, top_ids = torch.topk(probs, k=top_k)

print("PROMPT:", prompt)
print("\nTop przewidywania następnego tokenu:")
for prob, idx in zip(top_probs[0], top_ids[0]):
    token = tokenizer.decode([idx])
    print(f"{repr(token):>15} | p={prob.item():.4f}")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


PROMPT: Deep learning is

Top przewidywania następnego tokenu:
           ' a' | p=0.1623
         ' the' | p=0.0738
         ' not' | p=0.0472
          ' an' | p=0.0450
         ' one' | p=0.0312
        ' very' | p=0.0188
        ' what' | p=0.0135
   ' important' | p=0.0120
        ' also' | p=0.0116
        ' just' | p=0.0104


### Zadanie

1. Zmień prompt na:
   - `Warsaw is`
   - `Machine learning can`
   - `Najważniejszą cechą dobrego modelu jest`
2. Porównaj top-10 tokenów.
3. Zastanów się, kiedy wysoka pewność modelu jest uzasadniona, a kiedy bywa myląca.

## Parametry dekodowania

To samo wejście może prowadzić do bardzo różnych wyników w zależności od sposobu wyboru kolejnych tokenów.

Najważniejsze parametry:

- `max_new_tokens` - ile nowych tokenów model może dopisać,
- `do_sample=True` - czy losujemy z rozkładu, czy bierzemy najbardziej prawdopodobny token,
- `temperature` - jak „ostry” lub „rozlany” jest rozkład,
- `top_k` - losowanie tylko z `k` najlepszych tokenów,
- `top_p` - nucleus sampling, czyli losowanie z najmniejszego zbioru tokenów o łącznym prawdopodobieństwie `p`,
- `repetition_penalty` - kara za powtarzanie się,
- `num_return_sequences` - ile różnych odpowiedzi chcemy wygenerować.

In [ ]:
def compare_decoding(prompt, model_id="distilgpt2"):
    generator = pipeline("text-generation", model=model_id, tokenizer=model_id, device=device)

    settings = [
        {"name": "greedy", "do_sample": False},
        {"name": "temperature=0.3", "do_sample": True, "temperature": 0.3, "top_p": 0.95},
        {"name": "temperature=1.0", "do_sample": True, "temperature": 1.0, "top_p": 0.95},
        {"name": "top_k=20", "do_sample": True, "temperature": 0.8, "top_k": 20},
        {"name": "top_p=0.8", "do_sample": True, "temperature": 0.8, "top_p": 0.8},
    ]

    print("PROMPT:", prompt)
    for cfg in settings:
        name = cfg.pop("name")
        out = generator(prompt, max_new_tokens=40, return_full_text=True, **cfg)[0]["generated_text"]
        print("\n---", name, "---")
        print(out)

compare_decoding("Large language models can help with")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT: Large language models can help with


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- greedy ---
Large language models can help with the development of a language.




































...


--- temperature=0.3 ---
Large language models can help with the development of a language.




































Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- temperature=1.0 ---
Large language models can help with the application of the Python Language (GPL) as defined by the following:


























Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- top_k=20 ---
Large language models can help with the design and functionality of the language.

































--- top_p=0.8 ---
Large language models can help with the design of the language, and in particular, the development of new languages for the language. The authors discuss these concepts in more detail in their paper.











### Zadania

1. Uruchom kod dla kilku promptów:
   - technicznego,
   - codziennego,
   - po polsku.
2. Odpowiedz:
   - kiedy `greedy decoding` daje zbyt sztywne wyniki?
   - kiedy wysoka `temperature` zaczyna psuć sens?
   - kiedy `top_p` daje lepszy efekt niż `top_k`?

## 6. Tokenizacja i subtokeny w modelach decoderowych

Modele decoderowe również korzystają z tokenizacji subtokenowej. To szczególnie ważne, bo generacja odbywa się token po tokenie, więc sposób podziału tekstu bezpośrednio wpływa na:
- długość wejścia,
- koszt obliczeniowy,
- jakość generacji,
- łatwość obsługi nazw własnych, skrótów i rzadkich słów.

Dla języków fleksyjnych, takich jak polski, liczba subtokenów bywa większa niż dla angielskiego. To może utrudniać generację i zwiększać liczbę kroków potrzebnych do zbudowania sensownej odpowiedzi.

In [ ]:
def inspect_tokenization(model_id, text):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokens = tokenizer.tokenize(text)
    ids = tokenizer(text)["input_ids"]

    print(f"\nMODEL: {model_id}")
    print("Tekst:", text)
    print("Liczba tokenów:", len(ids))
    print("Tokeny:", tokens[:80])

texts = [
    "Transformers changed natural language processing.",
    "Modele językowe generują tekst token po tokenie.",
    "Politechnika Warszawska prowadzi zajęcia z dużych modeli językowych.",
]

for t in texts:
    print("\n" + "=" * 100)
    for model_id in ["distilgpt2", "gpt2", "EleutherAI/pythia-160m"]:
        inspect_tokenization(model_id, t)



MODEL: distilgpt2
Tekst: Transformers changed natural language processing.
Liczba tokenów: 7
Tokeny: ['Transform', 'ers', 'Ġchanged', 'Ġnatural', 'Ġlanguage', 'Ġprocessing', '.']

MODEL: gpt2
Tekst: Transformers changed natural language processing.
Liczba tokenów: 7
Tokeny: ['Transform', 'ers', 'Ġchanged', 'Ġnatural', 'Ġlanguage', 'Ġprocessing', '.']

MODEL: EleutherAI/pythia-160m
Tekst: Transformers changed natural language processing.
Liczba tokenów: 7
Tokeny: ['Transform', 'ers', 'Ġchanged', 'Ġnatural', 'Ġlanguage', 'Ġprocessing', '.']


MODEL: distilgpt2
Tekst: Modele językowe generują tekst token po tokenie.
Liczba tokenów: 19
Tokeny: ['Mode', 'le', 'Ġj', 'Ä', 'Ļ', 'zyk', 'owe', 'Ġgener', 'uj', 'Ä', 'ħ', 'Ġte', 'k', 'st', 'Ġtoken', 'Ġpo', 'Ġtoken', 'ie', '.']

MODEL: gpt2
Tekst: Modele językowe generują tekst token po tokenie.
Liczba tokenów: 19
Tokeny: ['Mode', 'le', 'Ġj', 'Ä', 'Ļ', 'zyk', 'owe', 'Ġgener', 'uj', 'Ä', 'ħ', 'Ġte', 'k', 'st', 'Ġtoken', 'Ġpo', 'Ġtoken', 'ie', '.']


### Pytania

1. Który tokenizer daje najmniej tokenów?
2. Czy polskie zdania rozpadają się na więcej subtokenów niż angielskie?
3. Jak wpływa to na koszt generacji i jakość odpowiedzi?

## 7. Prompting - czyli jak rozmawiać z decoderem

Model decoderowy jest bardzo wrażliwy na formę wejścia. Nawet niewielka zmiana promptu może diametralnie zmienić wynik.

Dobre prompty zwykle:
- precyzują zadanie,
- określają format odpowiedzi,
- zawierają kontekst,
- ograniczają długość lub styl odpowiedzi.

Porównajmy trzy wersje tego samego zadania.

In [ ]:
prompts = [
    "Explain overfitting.",
    "Explain overfitting in simple words.",
    "Explain overfitting in simple words to a first-year university student. Give 3 bullet points and one short example.",
]

generator = pipeline("text-generation", model="distilgpt2", tokenizer="distilgpt2", device=device)

for prompt in prompts:
    out = generator(prompt, max_new_tokens=80, do_sample=True, temperature=0.8)[0]["generated_text"]
    print("\n" + "=" * 100)
    print("PROMPT:", prompt)
    print("WYNIK:")
    print(out)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



PROMPT: Explain overfitting.
WYNIK:
Explain overfitting.

“We decided instead to put a lot of effort into designing and prototyping and packaging the game into a free-to-play game, which is why we decided to put a lot of effort into designing and packaging the game into a free-to-play game, which is why we decided to put a lot of effort into designing and packaging the game into a free-to-


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



PROMPT: Explain overfitting in simple words.
WYNIK:
Explain overfitting in simple words. A great design, a great way to express yourself.

PROMPT: Explain overfitting in simple words to a first-year university student. Give 3 bullet points and one short example.
WYNIK:
Explain overfitting in simple words to a first-year university student. Give 3 bullet points and one short example.




The reason this is not a simple solution, but it is one of the most important questions for a university institution in the world.


### Zadanie

Napisz 3 własne prompty dotyczące jednego tematu, ale o różnym poziomie precyzji:
- bardzo ogólny,
- średnio precyzyjny,
- bardzo precyzyjny.

Następnie porównaj:
- długość odpowiedzi,
- strukturę odpowiedzi,
- zgodność z poleceniem.

### Few-shot prompting

Decoder można też „naprowadzać” przykładami. Wtedy w samym prompcie pokazujemy, jakiego typu odpowiedzi oczekujemy.

To nie jest trenowanie modelu - to tylko sprytne przygotowanie wejścia.

In [ ]:
few_shot_prompt = '''
Poniżej znajdują się przykłady klasyfikacji sentymentu.

Tekst: To był świetny film.
Etykieta: pozytywny

Tekst: Zupełnie stracony czas.
Etykieta: negatywny

Tekst: Aktorstwo było poprawne, ale fabuła nudna.
Etykieta: negatywny

Tekst: Ten serial ma świetne dialogi i bardzo dobre tempo.
Etykieta:
'''.strip()

generator = pipeline("text-generation", model="distilgpt2", tokenizer="distilgpt2", device=device)
out = generator(few_shot_prompt, max_new_tokens=12, do_sample=False)[0]["generated_text"]

print(out)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=12) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Poniżej znajdują się przykłady klasyfikacji sentymentu.

Tekst: To był świetny film.
Etykieta: pozytywny

Tekst: Zupełnie stracony czas.
Etykieta: negatywny

Tekst: Aktorstwo było poprawne, ale fabuła nudna.
Etykieta: negatywny

Tekst: Ten serial ma świetne dialogi i bardzo dobre tempo.
Etykieta: negatywny
Tekst: ś


### Pytania

1. Czy model zrozumiał schemat zadania?
2. Co się stanie, gdy przykłady będą niespójne?
3. Jak few-shot prompting różni się od prawdziwego fine-tuningu?

## 8. Ograniczenia modeli decoderowych

Choć modele decoderowe świetnie generują tekst, mają też typowe problemy:

- halucynacje,
- zmyślanie faktów i źródeł,
- nadmierna pewność,
- powtarzanie tokenów,
- uciekanie od tematu,
- przechodzenie na inny język,
- wrażliwość na prompt.

To, że odpowiedź brzmi płynnie, nie oznacza jeszcze, że jest poprawna.

### Zadanie

Spróbuj „zepsuć” model:
1. zadaj bardzo niejasne pytanie,
2. poproś o szczegółowe fakty bez kontekstu,
3. użyj promptu po polsku z modelem głównie anglojęzycznym,
4. ustaw bardzo wysoką temperaturę.

Zapisz, jakie błędy pojawiają się najczęściej.

## 9. Krótkie dostrajanie modelu decoderowego

Poniżej znajduje się **minimalny** przykład dostrajania modelu typu decoder do prostego zadania generacyjnego.

Nie będziemy trenować wielkiego modelu od zera. Zamiast tego:
- przygotujemy mały zbiór prompt-odpowiedź,
- zbudujemy z niego sekwencje tekstowe,
- wykonamy bardzo krótki fine-tuning małego modelu.

To ćwiczenie ma charakter dydaktyczny - chodzi o zrozumienie procesu, a nie osiągnięcie produkcyjnej jakości.

In [ ]:
train_examples = [
    {
        "prompt": "Pytanie: Co trzeba oddać, aby zaliczyć laboratorium?\nOdpowiedź:",
        "answer": " Aby zaliczyć laboratorium, trzeba oddać kompletny notebook."
    },
    {
        "prompt": "Pytanie: Kiedy odbywają się konsultacje?\nOdpowiedź:",
        "answer": " Konsultacje odbywają się w środy o 14:00."
    },
    {
        "prompt": "Pytanie: Czy na zajęciach można pracować w parach?\nOdpowiedź:",
        "answer": " Tak, na zajęciach można pracować w parach."
    },
    {
        "prompt": "Pytanie: W jakim środowisku wykonujemy zadania?\nOdpowiedź:",
        "answer": " Zadania wykonujemy w środowisku Google Colab."
    },
    {
        "prompt": "Pytanie: Do kiedy należy oddać projekt końcowy?\nOdpowiedź:",
        "answer": " Projekt końcowy należy oddać do 30 czerwca."
    },
]

df = pd.DataFrame(train_examples)
df

,prompt,answer
0,"Pytanie: Co trzeba oddać, aby zaliczyć laborat...","Aby zaliczyć laboratorium, trzeba oddać kompl..."
1,Pytanie: Kiedy odbywają się konsultacje?\nOdpo...,Konsultacje odbywają się w środy o 14:00.
2,Pytanie: Czy na zajęciach można pracować w par...,"Tak, na zajęciach można pracować w parach."
3,Pytanie: W jakim środowisku wykonujemy zadania...,Zadania wykonujemy w środowisku Google Colab.
4,Pytanie: Do kiedy należy oddać projekt końcowy...,Projekt końcowy należy oddać do 30 czerwca.


In [ ]:
dataset = Dataset.from_pandas(df)

model_id = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def build_text(example):
    return {"text": example["prompt"] + example["answer"]}

dataset = dataset.map(build_text)
dataset

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Dataset({
    features: ['prompt', 'answer', 'text'],
    num_rows: 5
})

### Tokenizacja danych treningowych

In [ ]:
max_length = 96

def tokenize_function(example):
    encoded = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=max_length,
    )
    encoded["labels"] = encoded["input_ids"].copy()
    return encoded

tokenized = dataset.map(tokenize_function)
tokenized = tokenized.remove_columns([col for col in tokenized.column_names if col not in ["input_ids", "attention_mask", "labels"]])
tokenized.set_format("torch")
tokenized[0]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

{'input_ids': tensor([20519,    83, 34166,    25,  1766,   491, 38130,    64,  5629,    64,
         38325,    11,   450,    88,  1976,   282,   291,  7357, 38325,  4827,
         30732,    30,   198,    46, 26059,   322,   798,   129,   118,    25,
           317,  1525,  1976,   282,   291,  7357, 38325,  4827, 30732,    11,
           491, 38130,    64,  5629,    64, 38325,   479,   296, 37069,  3281,
         20922,    13, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256]),
 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0

### Wczytanie modelu

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_id)

model.config.pad_token_id = tokenizer.pad_token_id

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Argumenty treningu

W tym kroku definiujemy parametry treningu modelu.

Obiekt `TrainingArguments` określa najważniejsze ustawienia procesu uczenia, takie jak:

* gdzie zapisać wyniki,
* ile epok ma trwać trening,
* jaki batch size wykorzystać,
* jaka ma być wartość learning rate.

W tym przykładzie:

* model trenujemy przez `30` epok,
* używamy małego batch size `2`,
* nie zapisujemy checkpointów ani ewaluacji w trakcie,
* logi pojawiają się po każdej epoce.

Następnie tworzymy obiekt `Trainer`, który odpowiada za uruchomienie treningu.

Przekazujemy do niego:

* model,
* ustawienia treningu,
* dane treningowe,
* `data_collator`, który przygotowuje batch danych dla modelu decoderowego.

Ważne jest ustawienie `mlm=False`, ponieważ nie trenujemy modelu typu masked language model, tylko model autoregresyjny, przewidujący kolejne tokeny.

In [ ]:
args = TrainingArguments(
    output_dir="decoder_ft_model",
    eval_strategy="no",
    save_strategy="no",
    logging_strategy="epoch",
    num_train_epochs=30,
    per_device_train_batch_size=2,
    learning_rate=5e-5,
    weight_decay=0.01,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

### Trening

In [ ]:
trainer.train()

Step,Training Loss
3,4.768993
6,4.225259
9,3.629702
12,3.093175
15,2.590049
18,2.306259
21,1.952905
24,1.812446
27,1.554044
30,1.310527


TrainOutput(global_step=90, training_loss=1.2606305195225609, metrics={'train_runtime': 5.263, 'train_samples_per_second': 28.501, 'train_steps_per_second': 17.1, 'total_flos': 3674485555200.0, 'train_loss': 1.2606305195225609, 'epoch': 30.0})

### Test po krótkim dostrojeniu

In [ ]:
generator = pipeline(
    "text-generation",
    model=trainer.model,
    tokenizer=tokenizer,
    device=device
)

test_prompts = [
    "Pytanie: Co trzeba oddać, aby zaliczyć laboratorium?\nOdpowiedź:",
    "Pytanie: Czy na zajęciach można pracować w parach?\nOdpowiedź:",
    "Pytanie: W jakim środowisku wykonujemy zadania?\nOdpowiedź:",
]

for prompt in test_prompts:
    out = generator(
        prompt,
        max_new_tokens=30,
        do_sample=False,
        return_full_text=True
    )[0]["generated_text"]
    print("\n" + "=" * 100)
    print(out)

Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Pytanie: Co trzeba oddać, aby zaliczyć laboratorium?
Odpowiedź: Aby zaliczyć laboratorium, trzeba oddać kompletny notebook.
Odpowiedź:

Pytanie: Czy na zajęciach można pracować w parach?
Odpowiedź: Projekt końna pracować w parach.
Odpowiedź: Projekt koń

Pytanie: W jakim środowisku wykonujemy zadania?
Odpowiedź: Zadania wykonujemy w środowisku Google Colab.
Odpowiedź: Zadania


## 11. Gradio, interfejs użytkownika
Gradio to biblioteka, która pozwala szybko zrobić prosty interfejs graficzny do modelu.

Dzięki temu nie musimy za każdym razem uruchamiać modelu ręcznie w kodzie.
Zamiast tego możemy:

- wpisać prompt w oknie,

- ustawić parametry,

- kliknąć i zobaczyć wynik.


Korzystając z [Quickstart gradio](https://www.gradio.app/guides/quickstart) wykonaj poniższe zadania.

### Zaimportuj bibliotekę

### Wczytanie wybrany model z huggingface

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

### Przygotowanie funkcji generowania odpowiedzi modelu

Przygotuj funkcję `generate_text(prompt, max_new_tokens, temperature, top_k, top_p)`, która jako argumenty przyjmuje:

* `prompt` — tekst wejściowy przekazywany do modelu,
* `max_new_tokens` — maksymalną liczbę nowych tokenów do wygenerowania,
* `temperature` — parametr sterujący losowością generowania,
* `top_k` — liczbę najbardziej prawdopodobnych tokenów branych pod uwagę podczas generowania,
* `top_p` — próg prawdopodobieństwa stosowany w metodzie nucleus sampling.

Funkcja powinna przekazać tekst wejściowy do tokenizatora, wygenerować odpowiedź modelu z użyciem podanych parametrów, a następnie **zwrócić wygenerowany tekst w postaci ciągu znaków**.


In [ ]:
def generate_text(prompt, max_new_tokens, temperature, top_k, top_p):

    return None

Przygotuj interfejs użytkownika w bibliotece **Gradio**, który będzie wykorzystywał wcześniej zdefiniowaną funkcję `generate_text` do generowania tekstu.

Interfejs powinien zawierać:

* pole tekstowe `Prompt` umożliwiające wpisanie tekstu wejściowego,
* suwak `max_new_tokens` do określenia maksymalnej liczby generowanych tokenów,
* suwak `temperature` do ustawienia poziomu losowości generowania,
* suwak `top_k` do ograniczenia liczby branych pod uwagę tokenów,
* suwak `top_p` do ustawienia progu prawdopodobieństwa w metodzie nucleus sampling,
* pole wynikowe `Wynik`, w którym zostanie wyświetlony wygenerowany tekst.

Interfejs powinien mieć w tytule wbrany model oraz zostać uruchomiony z możliwością udostępnienia publicznego linku przy pomocy `launch(share=True)`.


#

In [ ]:
demo = gr.Interface(
    fn=generate_text,
    inputs=[
        gr.Textbox(...),
        gr.Slider(...),
        gr.Slider(..),
        gr.Slider(...),
        gr.Slider(...),
    ],
    outputs=gr.Textbox(lines=10, label="Wynik"),
    title="Decodery, LLMy i generowanie tekstu"
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0c220adc67cfcb4a08.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
